In [ ]:
%%capture
!pip install unsloth trl peft accelerate bitsandbytes datasets huggingface_hub

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length = 1024, # T4: 2048 not 4096
    dtype = None,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

In [ ]:
from datasets import load_dataset
# option A: Drive
# from google.colab import drive; drive.mount('/content/drive')
# path = "/content/drive/MyDrive/F1-Dataset/qa.jsonl"
path = "/content/qa.jsonl" # if dragged to sidebar
ds = load_dataset("json", data_files=path)["train"].train_test_split(test_size=0.1, seed=42) # 9485/1054

def fmt(examples):
    return {"text": [tokenizer.apply_chat_template(
        [{"role":"user","content": q},{"role":"assistant","content": a}],
        tokenize=False, add_generation_prompt=False) for q,a in zip(examples["question"], examples["answer"])]}
ds = ds.map(fmt, batched=True)
print(ds)

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    dataset_text_field="text", max_seq_length=2048,
    args=SFTConfig(
        per_device_train_batch_size=4, # T4 limit
        gradient_accumulation_steps=2, # eff 8
        warmup_ratio=0.03, num_train_epochs=1.0, # fix: not 3.0
        learning_rate=2e-4, packing=True, fp16=True, logging_steps=10,
        eval_steps=200, save_steps=200, output_dir="outputs",
        optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine",
        seed=3407, report_to="none",
    ),
)
trainer.train() # ~90 min T4. Stop if eval_loss rises after ~800 steps

In [ ]:
FastLanguageModel.for_inference(model)
def ask(q):
    prompt = tokenizer.apply_chat_template([{"role":"user","content": q}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=80, temperature=0.2, top_p=0.9, repetition_penalty=1.1, use_cache=True)
    print(tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1])




ask("Who is the only driver to have won 5 World Drivers' Championships?")
ask("How many drivers have won 5 titles?")

